# Exploratory Data Analysis

This notebook previews the raw data and per-instance model scores for the following benchmarks:
- [Chatbot Arena](https://huggingface.co/datasets/lmsys/chatbot_arena_conversations)
- [Chatbot Arena (New)](https://huggingface.co/datasets/lmarena-ai/arena-human-preference-100k)
- [DS-1000](https://huggingface.co/datasets/xlangai/DS-1000)
- [MATH](https://huggingface.co/datasets/EleutherAI/hendrycks_math)
- [MMLU](https://huggingface.co/datasets/cais/mmlu) 
- [WildChat10K](https://huggingface.co/datasets/allenai/WildChat-1M) (a curated subset of WildChat)

Import packages.

In [27]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from collections import Counter
from src.utils.enum import Dataset, MetadataKey  
from src.utils.data import load_dataset
from src.utils.model import load_model_scores

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Datasets

Load each dataset and view some of their instances.

In [28]:
datasets = [d.value for d in Dataset]

for dataset in datasets:
    # Skip Chatbot Arena (New) because it wasn't downloaded
    if dataset == Dataset.CHATBOT_ARENA_NEW:
        continue

    df = load_dataset(dataset)

    print(f"{dataset}: {len(df)} instances")
    display(df.head())
    print("-" * 200)

Chatbot-Arena: 44230 instances


,instruction,response
0,explain function calling. how would you call a...,Function calling is the process of invoking a ...
1,How can I create a test set for a very rare ca...,Creating a test set for a very rare category c...
2,What is the best way to travel from Tel-Aviv t...,The best way to travel from Tel-Aviv to Jerusa...
3,"Construct a rap battle, in the style of Epic R...","(Verse 1 - Zeus)\n\nI'm the king of the gods, ..."
4,Why water is not used in bath tub?,Water is actually used in a bath tub. A bath t...


--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
DS-1000: 1000 instances


,prompt,reference_code,metadata,code_context
0,Problem:\nI have the following DataFrame:\n ...,"def g(df, List):\n return df.iloc[List]\n\n...","{'problem_id': 0, 'library_problem_id': 0, 'li...",import pandas as pd\nimport numpy as np\nimpor...
1,Problem:\nI have the following DataFrame:\n ...,"def g(df, List):\n df2 = df.iloc[List].rein...","{'problem_id': 1, 'library_problem_id': 1, 'li...",import pandas as pd\nimport numpy as np\nimpor...
2,Problem:\nI have following pandas dataframe :\...,def g(df):\n return df.where(df.apply(lambd...,"{'problem_id': 2, 'library_problem_id': 2, 'li...",import pandas as pd\nimport numpy as np\nimpor...
3,Problem:\nI have following pandas dataframe :\...,def g(df):\n return df.where(df.apply(lambd...,"{'problem_id': 3, 'library_problem_id': 3, 'li...",import pandas as pd\nimport numpy as np\nimpor...
4,Problem:\nI have following pandas dataframe :\...,result = df.where(df.apply(lambda x: x.map...,"{'problem_id': 4, 'library_problem_id': 4, 'li...",import pandas as pd\nimport numpy as np\nimpor...


--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
MATH: 5000 instances


...

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
MMLU: 14042 instances


,question,subject,choices,answer
0,Find the degree for the given field extension ...,abstract_algebra,"[0, 4, 2, 6]",1
1,"Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the i...",abstract_algebra,"[8, 2, 24, 120]",2
2,Find all zeros in the indicated finite field o...,abstract_algebra,"[0, 1, 0,1, 0,4]",3
3,Statement 1 | A factor group of a non-Abelian ...,abstract_algebra,"[True, True, False, False, True, False, False,...",1
4,Find the product of the given polynomials in t...,abstract_algebra,"[2x^2 + 5, 6x^2 + 4x + 6, 0, x^2 + 1]",1


--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
WildChat10K: 10000 instances


,instruction,response
0,1. Smart Rings Thermostats Digital Body Temper...,1. Smart Rings Thermostats - Keep yourself com...
1,Write the following as a story. Beidou walks i...,"Once upon a time in the land of Teyvat, there ..."
2,What voting system was employed in the Comita ...,"The Comitia Centuriata, also known as the Cent..."
3,check and rewrite cover letter: I am writing t...,"Dear Hiring Manager,\n\nI am writing to expres..."
4,If my guinea pig almost stopped eating solid f...,If your guinea pig has almost stopped eating s...


--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


**Notes**
- Number of instances:
  - The local file for Chatbot Arena has more instances than the version on HuggingFace
    - Also, the file doesn't contain any info  that can be used to identify the instances (e.g., question IDs)
  - The local files for DS-1000, MATH, and MMLU have the same number of instances as the versions on HuggingFace
  - The local file for WildChat10K has the expected number of instances
    - However, the file doesn't contain any info that can be used to identify which instances were used from the full dataset (e.g., conversation hashes, hashed IPs, etc.)
- Subsets:
  - The local files for Chatbot Arena and WildChat10K don't contain pre-defined subsets — they only contains questions and answers
  - The "metadata" column in DS-1000 could be potentially used to categorize the instances
  - MATH and MMLU come with pre-defined subsets

### Subsets

View MATH's subsets and the number of instances in each one.

In [29]:
df = load_dataset(Dataset.MATH)
subset_col = "subset"
df = df[subset_col].value_counts().reset_index(name="count")

print(f"Number of {subset_col}s: {len(df)}")
df.head()

Number of subsets: 7


,subset,count
0,algebra,1187
1,intermediate_algebra,903
2,prealgebra,871
3,precalculus,546
4,number_theory,540


View MMLU's subsets and the number of instances in each one.

In [30]:
df = load_dataset(Dataset.MMLU)
subset_col = "subject"
df = df[subset_col].value_counts().reset_index(name="count")

print(f"Number of {subset_col}s: {len(df)}")
df.head()

Number of subjects: 57


,subject,count
0,professional_law,1534
1,moral_scenarios,895
2,miscellaneous,783
3,professional_psychology,612
4,high_school_psychology,545


#### DS-1000

View DS-1000's `metadata` column to see if there are any potential ways to split the instances into subsets.

In [31]:
ds_1000_df = load_dataset(Dataset.DS_1000)
metadata_col = ds_1000_df["metadata"].to_list()
metadata_col[0]

{'problem_id': 0,
 'library_problem_id': 0,
 'library': 'Pandas',
 'test_case_cnt': 1,
 'perturbation_type': 'Origin',
 'perturbation_origin_id': 0}

`library` and `perturbation_type` seem like two potential ways to split the instances into subsets. Also, the [DS-1000 GitHub repo](https://github.com/xlang-ai/DS-1000/?tab=readme-ov-file#usage) says that the possible values for `library` and their counts are: 

| lib        | count |
|------------|-------|
| Matplotlib | 155   |
| Numpy      | 220   |
| Pandas     | 291   |
| Pytorch    | 68    |
| Scipy      | 106   |
| Sklearn    | 115   |
| Tensorflow | 45    |

View the unique values in the `library` column and their counts.

In [32]:
key = MetadataKey.LIBRARY
counts = Counter(metadata_dict[key] for metadata_dict in metadata_col)

# Sort by key in alphabetical order
counts_df = pd.DataFrame(
    sorted(counts.items(), key=lambda x: x[0]),
    columns=[key, "count"],
)

display(counts_df)

,library,count
0,Matplotlib,155
1,Numpy,220
2,Pandas,291
3,Pytorch,68
4,Scipy,106
5,Sklearn,115
6,Tensorflow,45


View the unique values in the `perturbation_type` column and their counts.

In [33]:
key = MetadataKey.PERTURBATION_TYPE
counts = Counter(metadata_dict[key] for metadata_dict in metadata_col)

# Sort by key in alphabetical order
counts_df = pd.DataFrame(
    sorted(counts.items(), key=lambda x: x[0]),
    columns=[key, "count"],
)

display(counts_df)

,perturbation_type,count
0,Difficult-Rewrite,162
1,Origin,452
2,Semantic,234
3,Surface,152


It looks like **both** `library` and `perturbation_type` are promising ways to categorize instances in DS-1000.

## Evaluation Results

Only DS-1000, MATH, and MMLU have per-instance model scores. Filter `datasets` so it only contains those datasets.

In [34]:
datasets = [Dataset.DS_1000, Dataset.MATH, Dataset.MMLU]

View each dataset's evaluation results.

In [35]:
for dataset in datasets:
    model_scores_df = load_model_scores(dataset)
    print(f"{dataset}: {len(model_scores_df)} instances")
    display(model_scores_df.head())

DS-1000: 1000 instances


,deepseek-coder-6.7b-base,gpt-3.5-turbo-0613,gpt-4o-2024-08-06
0,1,0,0
1,0,1,1
2,1,0,1
3,1,1,1
4,0,1,0


MATH: 5000 instances


,dart-math-llama3-8b-uniform,deepseek-math-7b-instruct,gpt-4o-mini-2024-07-18,Llama-3-8B-Instruct,Llama-3.1-8B-Instruct
0,1,1,1,1,1
1,1,1,1,0,1
2,0,1,1,1,0
3,1,0,1,0,1
4,1,1,1,1,1


MMLU: 14042 instances


,claude-3.5-haiku,gpt-3.5-turbo,gpt-4o-mini-2024-07-18,Llama-3.1-70B-Instruct,Llama-3.1-8B-Instruct,Llama-3.1-Tulu-3-70B,Llama-3.1-Tulu-3-8B,Qwen2.5-72B-Instruct,Qwen2.5-7B-Instruct
0,1,0,1,1,0,1,0,1,0
1,0,0,0,1,0,0,0,1,0
2,1,1,0,1,0,0,0,0,1
3,0,0,0,0,0,1,0,1,1
4,1,0,1,1,1,1,0,1,1
